# RFW aligned-BIN origin embedding extraction

RFW official 10-fold pair protocol과 aligned BIN의 순서를 검증한 뒤, 선택한 FR checkpoint로 48,000 pair occurrences의 512D 원본 임베딩을 한 번만 추출한다. 이 단계는 PCA/PQ를 fit하지 않으며 RFW를 open-set 데이터셋으로 재해석하지 않는다.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("C:/ronbun project root could not be located")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.datasets import inspect_rfw_aligned_bin_archive
from research.experiments import extract_rfw_origin_embeddings


## 실행 설정

`ARTIFACT_STORAGE_MODE`는 기존 공통 runner 계약을 유지한다. `results_only`는 재사용 가능한 결과 artifact 아래에 저장하고, `full`은 run artifact 아래에 저장한다. 기본 checkpoint는 Step 7에서 검증한 EdgeFace-XS γ=0.6이다.


In [ ]:
MODEL_UID = "edgeface-a348c305af33c223b337"
MODEL_SPEC_PATH = PROJECT_ROOT / "runs/step2/model_registry" / f"{MODEL_UID}.json"
PAIR_PROTOCOL_PATH = PROJECT_ROOT / "data/interim/rfw/pair_protocol.csv"
ALIGNED_BIN_ARCHIVE = PROJECT_ROOT / "data/raw/RFW/bin_for_mxnet/RFW_test.tar.gz"
EXPECTED_ARCHIVE_SHA256 = "8259e53ab1b542a9747335f34b2ae48e22a675bce472e391d6b94bc900fde572"

DEVICE = "cuda"
BATCH_SIZE = 128
HORIZONTAL_FLIP_TTA = False
EXECUTE_STAGE = True
WRITE_OUTPUTS = True
REUSE_COMPLETED = True
ARTIFACT_STORAGE_MODE = "results_only"

if ARTIFACT_STORAGE_MODE not in {"results_only", "full"}:
    raise ValueError("ARTIFACT_STORAGE_MODE must be results_only or full")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True requires EXECUTE_STAGE=True")
ARTIFACT_ROOT = PROJECT_ROOT / ("results" if ARTIFACT_STORAGE_MODE == "results_only" else "runs")
OUTPUT_DIR = ARTIFACT_ROOT / "rfw_step7/origin_embeddings" / MODEL_UID
{
    "model_uid": MODEL_UID,
    "device": DEVICE,
    "horizontal_flip_tta": HORIZONTAL_FLIP_TTA,
    "artifact_storage_mode": ARTIFACT_STORAGE_MODE,
    "output_dir": str(OUTPUT_DIR),
}


## source/protocol preflight 및 추출

먼저 archive SHA와 4개 group의 12,000-image/6,000-pair 구성을 검증한다. 정식 실행에서는 `00_data_preparation.ipynb`가 만든 전체 `pair_protocol.csv`가 필요하다. 완료 artifact가 있으면 manifest와 모든 SHA가 일치할 때만 재사용한다.


In [ ]:
archive_summary = inspect_rfw_aligned_bin_archive(
    ALIGNED_BIN_ARCHIVE,
    expected_sha256=EXPECTED_ARCHIVE_SHA256,
    strict_official=True,
)
if not PAIR_PROTOCOL_PATH.is_file():
    raise FileNotFoundError(
        "Run notebooks/rfw/00_data_preparation/00_data_preparation.ipynb first"
    )
pairs = pd.read_csv(PAIR_PROTOCOL_PATH)
if len(pairs) != 24000:
    raise ValueError(f"official RFW evaluation requires 24,000 pairs, got {len(pairs)}")

def progress(message, details):
    print(message, details)

origin_artifact = None
if EXECUTE_STAGE and WRITE_OUTPUTS:
    origin_artifact = extract_rfw_origin_embeddings(
        aligned_bin_archive_path=ALIGNED_BIN_ARCHIVE,
        pairs=pairs,
        model_spec_path=MODEL_SPEC_PATH,
        output_dir=OUTPUT_DIR,
        expected_archive_sha256=EXPECTED_ARCHIVE_SHA256,
        expected_model_uid=MODEL_UID,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        horizontal_flip_tta=HORIZONTAL_FLIP_TTA,
        strict_official=True,
        reuse_completed=REUSE_COMPLETED,
        progress=progress,
    )
origin_artifact.manifest if origin_artifact else archive_summary


## 다음 단계

`rfw_origin_embedding_manifest.json`, `_SUCCESS`, 배열 및 CSV의 SHA를 확인한 후 `notebooks/rfw/02_compression/00_rfw_frozen_codec_verification.ipynb`로 진행한다. RFW 임베딩으로 codec을 재학습하지 않는다.
